In [ ]:
# C1QA, C1QC and LYVE1 expression versus Palantir pseudotime per cluster.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl
import seaborn as sns

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/8_figures"
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
genes = ["C1QA", "C1QC", "LYVE1"]
expression_data = adata[:, genes].X.toarray()  # Convert sparse matrix to dense if needed

# Create DataFrame
df = pd.DataFrame(expression_data, columns=genes, index=adata.obs.index)
df['pseudotime'] = adata.obs['palantir_pseudotime'].values
df['cluster'] = adata.obs['leiden_res_0_8'].values

# Group by cluster and calculate mean pseudotime and mean expression for each gene
aggregated_df = (
    df.groupby('cluster')
    .agg({**{gene: 'mean' for gene in genes}, 'pseudotime': 'mean'})
    .reset_index()
)

# Reshape to long format
plot_df = aggregated_df.melt(
    id_vars=['cluster', 'pseudotime'],
    var_name='gene',
    value_name='expression'
)

# Plot
plt.figure(figsize=(12, 8))
palette = sns.color_palette("husl", len(aggregated_df['cluster'].unique()))  # Custom palette

for gene in genes:
    subset = plot_df[plot_df["gene"] == gene]
    sns.scatterplot(
        x="pseudotime", 
        y="expression", 
        hue="cluster", 
        data=subset, 
        palette=palette, 
        s=100, 
        legend=None
    )
    sns.lineplot(
        x="pseudotime", 
        y="expression", 
        data=subset, 
        linestyle="--", 
        alpha=0.8, 
        label=f"{gene} smoothed"
    )

# Customize plot
plt.legend(title="Gene", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("Mean Pseudotime")
plt.ylabel("Mean Gene Expression")
plt.title("Gene Expression vs. Pseudotime per Cluster")
plt.tight_layout()
plt.show()


In [ ]:
# Plot
plt.figure(figsize=(12, 8))
palette = sns.color_palette("husl", len(aggregated_df['cluster'].unique()))  # Custom palette

# Scatterplot and lineplot for each gene
for gene in genes:
    subset = plot_df[plot_df["gene"] == gene]
    
    # Scatterplot for gene expression vs pseudotime, colored by cluster
    sns.scatterplot(
        x="pseudotime", 
        y="expression", 
        hue="cluster", 
        data=subset, 
        palette=palette, 
        s=100, 
        legend=False  # Do not add legend here
    )
    
    # Lineplot for smoothed gene expression
    sns.lineplot(
        x="pseudotime", 
        y="expression", 
        data=subset, 
        linestyle="--", 
        alpha=0.8, 
        label=f"{gene} smoothed"
    )

# Add a separate legend for clusters
plt.legend(title="Gene", bbox_to_anchor=(1.05, 1), loc='upper left')

# Add a legend for clusters
handles, labels = plt.gca().get_legend_handles_labels()
cluster_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) for i in range(len(aggregated_df['cluster'].unique()))]
cluster_labels = [f"Cluster {i}" for i in aggregated_df['cluster'].unique()]
plt.legend(handles=handles + cluster_handles, labels=labels + cluster_labels, title="Gene & Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')

# Customize plot
plt.xlabel("Mean Pseudotime")
plt.ylabel("Mean Gene Expression")
plt.title("Example for Figure 3K")
plt.tight_layout()
plt.show()
